<h2>Description</h2>

Dans ce code, nous allons établir un modèle afin de prédire le débit horaire sur les Champs Élysées.

Imports

In [109]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.inspection import permutation_importance

doc = 'champs_elysees.csv'

df_final = pd.read_csv('../datasets_axes_with_all_features/'+ doc, sep=';')

In [110]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8747 entries, 0 to 8746
Data columns (total 42 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Unnamed: 0                 8747 non-null   int64  
 1   Identifiant arc            8747 non-null   int64  
 2   Libelle                    8747 non-null   object 
 3   Date et heure de comptage  8747 non-null   object 
 4   Débit horaire              8197 non-null   float64
 5   Taux d'occupation          8182 non-null   float64
 6   Etat trafic                8747 non-null   object 
 7   Identifiant noeud amont    8747 non-null   int64  
 8   Libelle noeud amont        8747 non-null   object 
 9   Identifiant noeud aval     8747 non-null   int64  
 10  Libelle noeud aval         8747 non-null   object 
 11  Etat arc                   8747 non-null   object 
 12  Date debut dispo data      8747 non-null   object 
 13  Date fin dispo data        8747 non-null   objec

In [ ]:
df_final = df_final.copy()
df_final['Date et heure de comptage'] = pd.to_datetime(df_final['Date et heure de comptage'], errors='coerce')
df_final = df_final.sort_values('Date et heure de comptage').reset_index(drop=True)

for col in ['est_vacances', 'est_ferie', 'est_avant_ferie', 'est_pieton']:
    if col in df_final.columns:
        df_final[col] = pd.to_numeric(df_final[col], errors='coerce')

features = [
    'Température', 'duree prec (en min)', 'precipitations heure',
    'heure_sin', 'heure_cos', 'jour_sin', 'jour_cos', 'mois_sin', 'mois_cos', 
    'force moyenne vent (m/s)', 'jour_semaine', 'est_weekend', 'est_vacances', 'est_avant_vacances',
    'est_ferie', 'est_avant_ferie', 'ensoleillement (en min)', 'est_pieton'
]
target = 'Débit horaire'

mask_known   = df_final[target].notna()
mask_missing = df_final[target].isna()

X_known = df_final.loc[mask_known, features].copy()
y_known = df_final.loc[mask_known, target].astype(float)
X_missing = df_final.loc[mask_missing, features].copy()

numeric_features = [
    'Température', 'precipitations heure',
    'heure_sin', 'heure_cos', 'jour_sin', 'jour_cos', 'mois_sin', 'mois_cos',
    'force moyenne vent (m/s)', 'duree prec (en min)', 'est_weekend', 'est_vacances', 'est_avant_vacances',
    'est_ferie', 'est_avant_ferie', 'ensoleillement (en min)', 'est_pieton'
]
categorical_features = ['jour_semaine']

try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)  
except TypeError:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)         

preprocess = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))]), numeric_features),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', ohe)
        ]), categorical_features),
    ],
    remainder='drop'
)

model = HistGradientBoostingRegressor(
    loss='absolute_error',   
    max_depth=12,
    max_iter=400,
    early_stopping=False,
    random_state=42
)

pipe = Pipeline(steps=[('prep', preprocess), ('model', model)])

# ---------- 3) Split chronologique ----------
X_train, X_test, y_train, y_test = train_test_split(
    X_known, y_known, test_size=0.2, shuffle=False
)

# ---------- 4) Pondérations (férié / veille / piéton) ----------
W_FERIE   = 3.0
W_AVANT   = 1.5
W_PIETON  = 10
POST_SCALE_FERIE = 1.0

def make_weights(X_frame):
    w = np.ones(len(X_frame), dtype=float)
    is_ferie  = X_frame['est_ferie'].fillna(0).astype(int).to_numpy()
    is_avant  = X_frame['est_avant_ferie'].fillna(0).astype(int).to_numpy()
    is_pieton = X_frame['est_pieton'].fillna(0).astype(int).to_numpy()

    w[is_ferie == 1]  = W_FERIE
    w[is_avant == 1]  = np.maximum(w[is_avant == 1], W_AVANT)
    w[is_pieton == 1] = W_PIETON

    # Option : normalisation pour garder une échelle de perte comparable
    #w *= (len(w) / w.sum())
    return w

w_train = make_weights(X_train)

# ---------- 5) Entraînement ----------
pipe.fit(X_train, y_train, model__sample_weight=w_train)

# ---------- 6) Prédiction + post-ajustement éventuel ----------
y_pred = pipe.predict(X_test)

mask_ferie_test = X_test['est_ferie'].fillna(0).astype(int).to_numpy() == 1
mask_est_pieton_test = X_test['est_pieton'].fillna(0).astype(int).to_numpy() == 1
y_pred[mask_ferie_test] *= POST_SCALE_FERIE
#y_pred[mask_est_pieton_test] *= 0.5

# ---------- 7) Évaluation ----------
r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² : {r2:.3f}")
print(f"MAE : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")

# Diagnostics par sous-régimes
is_pieton_test = X_test['est_pieton'].fillna(0).astype(int) == 1
print(f"Part d'observations piéton (test) : {is_pieton_test.mean():.1%}")
if is_pieton_test.any():
    mae_pieton = mean_absolute_error(y_test[is_pieton_test], y_pred[is_pieton_test])
    print(f"MAE (jours piéton) : {mae_pieton:.2f} (n={is_pieton_test.sum()})")
    mae_non_pieton = mean_absolute_error(y_test[~is_pieton_test], y_pred[~is_pieton_test])
    print(f"MAE (jours non piéton) : {mae_non_pieton:.2f} (n={(~is_pieton_test).sum()})")

df_final['Débit_prédit'] = np.nan
df_final.loc[X_test.index, 'Débit_prédit'] = y_pred


R² : 0.774
MAE : 85.86
RMSE : 120.20
Part d'observations piéton (test) : 2.1%
MAE (jours piéton) : 152.80 (n=34)
MAE (jours non piéton) : 84.44 (n=1606)


In [112]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8747 entries, 0 to 8746
Data columns (total 43 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Unnamed: 0                 8747 non-null   int64         
 1   Identifiant arc            8747 non-null   int64         
 2   Libelle                    8747 non-null   object        
 3   Date et heure de comptage  8747 non-null   datetime64[ns]
 4   Débit horaire              8197 non-null   float64       
 5   Taux d'occupation          8182 non-null   float64       
 6   Etat trafic                8747 non-null   object        
 7   Identifiant noeud amont    8747 non-null   int64         
 8   Libelle noeud amont        8747 non-null   object        
 9   Identifiant noeud aval     8747 non-null   int64         
 10  Libelle noeud aval         8747 non-null   object        
 11  Etat arc                   8747 non-null   object        
 12  Date d

In [113]:
from sklearn.inspection import permutation_importance
import pandas as pd
import numpy as np
import plotly.express as px

# 1) Importance par permutation sur le pipeline complet (prétraitements inclus)
perm = permutation_importance(
    estimator=pipe,
    X=X_test,
    y=y_test,
    n_repeats=20,
    random_state=42,
    scoring='neg_root_mean_squared_error'  # cohérent avec votre RMSE
)

imp_df = (
    pd.DataFrame({
        'feature': features,
        'importance_mean': perm.importances_mean,
        'importance_std': perm.importances_std
    })
    .sort_values('importance_mean', ascending=False)
)

print(imp_df.head(20))

# 2) Bar chart Plotly (top 20)
topk = imp_df.head(20).sort_values('importance_mean', ascending=True)
fig = px.bar(
    topk,
    x='importance_mean', y='feature',
    error_x='importance_std',
    orientation='h',
    title='Importance par permutation — Top 20 (plus haut = plus influent)'
)
fig.update_layout(xaxis_title="Perte de performance (Δ RMSE, signe inversé)", yaxis_title="")
fig.show()


                     feature  importance_mean  importance_std
3                  heure_sin       170.383322        3.285352
4                  heure_cos       123.849847        4.375213
5                   jour_sin        51.629386        2.902282
17                est_pieton        12.741725        1.144740
9   force moyenne vent (m/s)         3.354851        0.496831
14                 est_ferie         2.911638        1.839261
11               est_weekend         2.608940        0.683838
0                Température         2.581163        0.639220
10              jour_semaine         2.447849        0.651603
6                   jour_cos         1.749287        0.465527
12              est_vacances         0.912312        0.759830
16   ensoleillement (en min)         0.805434        0.288685
2       precipitations heure         0.031297        0.155144
13        est_avant_vacances         0.004950        0.034570
15           est_avant_ferie        -0.038440        0.151105
1       

In [114]:
time_index = df_final.loc[X_test.index, 'Date et heure de comptage']

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=time_index,
    y=y_pred,
    mode='lines',
    name='Débit prédit'
))
fig.add_trace(go.Scatter(
    x=time_index,
    y=y_test,
    mode='lines',
    name='Débit réel'
))

fig.update_layout(
    title="Comparaison des débits (réel vs prédit)",
    xaxis_title="Date et heure",
    yaxis_title="Débit horaire (véh/h)",
    hovermode='x unified'
)

fig.show()

In [115]:
X_all = df_final.loc[:, features].copy()

predictions_all = pipe.predict(X_all)

serie = pd.Series(predictions_all, index=df_final.index)

commun = df_final[target].notna() & serie.notna()

predictions_with_know = serie.loc[commun].astype(float)

mae = mean_absolute_error(predictions_with_know, y_known)
rmse = np.sqrt(mean_squared_error(predictions_with_know, y_known))
r2   = r2_score(predictions_with_know, y_known)

print("Nombre de valeurs : " + str(len(predictions_with_know)))
print(f"R² : {r2:.3f}")
print(f"MAE : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")


Nombre de valeurs : 8197
R² : 0.854
MAE : 58.70
RMSE : 101.48


In [116]:


fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_final['Date et heure de comptage'],
    y=predictions_all,
    mode='lines',
    name='Débit prédit'
))
fig.add_trace(go.Scatter(
    x=df_final['Date et heure de comptage'],
    y=df_final['Débit horaire'],
    mode='lines',
    name='Débit réel'
))

fig.update_layout(
    title="Comparaison des débits (réel vs prédit)",
    xaxis_title="Date et heure",
    yaxis_title="Débit horaire (véh/h)",
    hovermode='x unified'
)

fig.show()

<h2>Taux d'occupation</h2>

In [117]:
target_occ = 'Taux d\'occupation'

features_occ = [
    'Température', 'duree prec (en min)', 'Débit_prédit',
    'heure_sin', 'heure_cos', 'mois_sin', 'mois_cos',
    'force moyenne vent (m/s)', 'mois', 'jour_sin',
    'est_ferie', 'est_avant_ferie', 'ensoleillement (en min)', 'est_pieton'
]

mask_occ = df_final[target_occ].notna() & df_final['Débit_prédit'].notna()
X_occ = df_final.loc[mask_occ, features_occ].copy()
y_occ = df_final.loc[mask_occ, target_occ].astype(float)

X_train_occ, X_test_occ, y_train_occ, y_test_occ = train_test_split(
    X_occ, y_occ, test_size=0.2, shuffle=False
)

numeric_features_occ = [c for c in features_occ if c != 'jour_semaine']
categorical_features_occ = []

try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)

preprocess_occ = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numeric_features_occ),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', ohe)
        ]), categorical_features_occ),
    ],
    remainder='drop'
)

model_occ = HistGradientBoostingRegressor(
    loss='poisson',
    max_depth=5,
    max_iter=40,
    early_stopping=False,
    random_state=42
)

pipe_occ = Pipeline(steps=[('prep', preprocess_occ), ('model', model_occ)])

pipe_occ.fit(X_train_occ, y_train_occ)

y_pred_occ = pipe_occ.predict(X_test_occ)

print("=== Performances taux d'occupation ===")
print(f"R²   : {r2_score(y_test_occ, y_pred_occ):.3f}")
print(f"MAE  : {mean_absolute_error(y_test_occ, y_pred_occ):.2f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test_occ, y_pred_occ)):.2f}")
print(f"nb test : {len(y_test_occ)}")


=== Performances taux d'occupation ===
R²   : 0.813
MAE  : 2.74
RMSE : 3.59
nb test : 328


In [118]:
perm = permutation_importance(
    estimator=pipe_occ,
    X=X_test_occ,
    y=y_test_occ,
    n_repeats=20,
    random_state=42,
    scoring='neg_root_mean_squared_error'  
)

imp_df = (
    pd.DataFrame({
        'feature': features_occ,
        'importance_mean': perm.importances_mean,
        'importance_std': perm.importances_std
    })
    .sort_values('importance_mean', ascending=False)
)

print(imp_df.head(20))

topk = imp_df.head(20).sort_values('importance_mean', ascending=True)
fig = px.bar(
    topk,
    x='importance_mean', y='feature',
    error_x='importance_std',
    orientation='h',
    title='Importance par permutation — Top 20 (plus haut = plus influent)'
)
fig.update_layout(xaxis_title="Perte de performance (Δ RMSE, signe inversé)", yaxis_title="")
fig.show()


                     feature  importance_mean  importance_std
2               Débit_prédit         3.210138        0.168935
3                  heure_sin         2.261610        0.173446
4                  heure_cos         0.830924        0.099304
9                   jour_sin         0.270011        0.060339
0                Température         0.057746        0.061857
12   ensoleillement (en min)         0.039928        0.031140
5                   mois_sin         0.014131        0.029319
7   force moyenne vent (m/s)         0.006880        0.042973
6                   mois_cos         0.000000        0.000000
8                       mois         0.000000        0.000000
10                 est_ferie         0.000000        0.000000
11           est_avant_ferie         0.000000        0.000000
13                est_pieton         0.000000        0.000000
1        duree prec (en min)        -0.015657        0.017401


In [119]:
time_index_occ = df_final.loc[X_test_occ.index, 'Date et heure de comptage']

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=time_index_occ,
    y=y_pred_occ,
    mode='lines',
    name='Débit prédit'
))
fig.add_trace(go.Scatter(
    x=time_index_occ,
    y=y_test_occ,
    mode='lines',
    name='Débit réel'
))

fig.update_layout(
    title="Comparaison des débits (réel vs prédit)",
    xaxis_title="Date et heure",
    yaxis_title="Débit horaire (véh/h)",
    hovermode='x unified'
)

fig.show()